# Õppetund 17 - Kohalike AI-agentide loomine Foundry Locali ja Qweniga

Selles märkmikus ehitate **kohaliku inseneri abilise**, mis töötab täielikult teie tööjaamas. Pilvinõuet ei kasutata mitte kunagi. Abiline suudab:

1. **Kõnetada tööriistu** Qweni funktsioonikõne kaudu Foundry Localiga.
2. **Lugeda ja kuvada faile** liivakasti projektikataloogis.
3. **Analüüsida koodi** lihtsate mõõdikutega.
4. **Otsida dokumentatsioonist** lokaalsete RAG-i (Chroma) abil.
5. **Kasutada kohalikku MCP-serverit** (jätab viisakalt vahele, kui ühtegi pole seadistatud).

Agendi kood näeb välja peaaegu identselt pilveõpetustega — ainus erinevus on klientpunktil, mis liigub pilvest `localhost`-i.


## Seadistamine

Enne selle märkmiku käivitamist:

1. **Installi Microsoft Foundry Local** (vaata [dokumentatsiooni](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) oma opsüsteemi jaoks).
2. **Laadi alla ja käivita Qweni mudel:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Paigalda alljärgnevad Python paketid.

Kõik töötab lokaalselt. Masin umbes 8 GB RAM-iga on realistlik minimaalne.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Ühenda Foundry Localiga

`FoundryLocalManager` laeb mudeli alla, kui vaja, käivitab kohaliku teenuse ja annab meile **OpenAI-ga ühilduva lõpp-punkti**. Seejärel suuname tavalise OpenAI SDK sinna. API-võti on kohalik kohatäide — pilve tõendust ei kasutata.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Kohalikud tööriistad (liivakastitud failitoimingud)

Me loome kettale väikese näidisprojekti, seejärel määratleme tööriistad, mis on piiratud selle projekti juurkaustaga. Liivakasti kontroll on oluline ka kohapeal: tööriist, mis loeb suvalisi radu, käivitub teie kasutaja õigustes ja võib puudutada kõike, mida teie kasutaja saab.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Kohalik RAG koos Chromaga

Me sisestame väikese komplekti dokumentatsiooni katkenditest kohalikku Chroma kollektsiooni. Chroma töötab protsessis ja salvestab vektorid kettale — mitte serverit ega pilve. Tööriist `search_docs` hangib päringu jaoks kõige asjakohasemad katked.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Tööriista kutsumise tsükkel

Nüüd registreerime tööriistad mudeliga, kasutades OpenAI tööriistade skeemi, ning käivitame standardse tööriista kutsumise tsükli — mudel nõuab tööriista, me täidame selle kohapeal, edastame tulemuse tagasi ja kordame kuni mudel annab lõpliku vastuse. Qweni usaldusväärne funktsioonikõne võimaldab selle seadmes toimida.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Kohalik MCP (valikuline)

MCP on transport, mitte pilveteenus — MCP server võib töötada kohalikuna protsessina üle `stdio`. Allolev lahter näitab, kuidas ühenduda kohaliku MCP serveriga, kui sul on see seadistatud (näiteks failisüsteemi server). Kui `LOCAL_MCP_COMMAND` ei ole määratud, jätab see selle viisakalt vahele, nii et märkmik jookseb ikkagi lõpuni läbi.

Turvalisuse märkus: kohalik MCP server töötab sinu kasutaja õigustega. Piira see projektikaustale ja kontrolli selle väljundeid enne nende põhjal tegutsemist.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Kokkuvõte

Sa lõid inseneri abilise, mis töötab täielikult sinu masinas:

- **Foundry Local** pakkus **Qweni** mudelit OpenAI-ühilduva lõpp-punkti taga — nii et agendi kood vastab pilve õppetundidele.
- **Piiratud tööriistad** andsid agendile failidele ligipääsu ja koodi analüüsi ilma, et oleks vaja projektikaustast väljuda.
- **Chroma** pakkus **kohalikku RAG-i** dokumentatsiooni üle.
- **Local MCP** näitas, kuidas MCP ökosüsteemi võrguühenduseta taaskasutada.

Ühtegi pilvepõhist järeldust ei kasutatud.

### Väljakutse

Laienda kohalikku agenti, et:

1. **Töötab mitme MCP serveriga** — ühenda failisüsteemi server ja git server ning lase agendil nende vahel valida.
2. **Kasuta kohalikku mälu** — säilita lühike vestluse ajalugu kettal nii, et assistent mäletaks varasemaid kordusi ka märkmiku taaskäivitamisel.
3. **Tugevda kohalikku multiagendi orkestreerimist** — lisa teine kohalik agent (nt hindaja) ja lase kahel koos ülesannet lahendada.

Järgmises õppetükis õpid, kuidas turvaliselt juurutada AI-agente.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
